# Part 2 — Portfolio & Risk Analysis

1. **Portfolio health & trajectory** — how big is the book, is it growing, is it profitable, is credit quality stable?
2. **Risk segmentation** — which parts of the portfolio underperform, and what do they look like?

Input: `loans_clean.csv` from Part 1 (run the Part 1 notebook first, in the same Colab session).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

loans_clean = pd.read_csv("loans_clean.csv")
loans_clean["origination_date"] = pd.to_datetime(loans_clean["origination_date"])
print(loans_clean.shape)

### Unit economics per loan
What Affirm earns on each loan, after losses and funding (assumptions from Part 1, section 4):
- **revenue** = merchant fee (`mdr_fees`) + consumer interest (`interest_collected`)
- **funding cost** = `loan_amount × cost_of_funds_bps / 10,000 × term in years × 0.5`
  (loans are repaid gradually, so on average ~half the principal is outstanding; Split Pay term = 1.5 months)
- **contribution** = revenue − credit loss − funding cost

Everything is shown **as a % of GMV** (loan dollars) so segments of different sizes can be compared.

In [ ]:
is_split = loans_clean["product_type"] == "split_pay"
term_years = np.where(is_split, 1.5, loans_clean["loan_term_months"]) / 12

loans_clean["revenue"]      = loans_clean["mdr_fees"] + loans_clean["interest_collected"]
loans_clean["funding_cost"] = loans_clean["loan_amount"] * loans_clean["cost_of_funds_bps"] / 10000 * term_years * 0.5
loans_clean["contribution"] = loans_clean["revenue"] - loans_clean["credit_loss"] - loans_clean["funding_cost"]

# Helper columns for segmenting
loans_clean["bad"] = loans_clean["loan_status"].isin(["delinquent_30", "delinquent_60", "charged_off"])
loans_clean["quarter"] = loans_clean["origination_date"].dt.to_period("Q").astype(str)
loans_clean["fico_band"] = pd.cut(loans_clean["fico_score"], [299, 579, 639, 699, 759, 850],
                            labels=["<580", "580-639", "640-699", "700-759", "760+"]).astype(str)
loans_clean.loc[loans_clean["fico_score"].isna(), "fico_band"] = "Unknown"
loans_clean["customer"] = np.where(loans_clean["is_new_to_affirm"], "new", "repeat")

In [ ]:
def summarize(df, by):
    """One row per segment: size, credit quality and unit economics (% of GMV)."""
    g = df.groupby(by).agg(loans=("loan_id", "count"),
                           gmv=("loan_amount", "sum"),
                           avg_fico=("fico_score", "mean"),
                           bad_rate=("bad", "mean"),
                           revenue=("revenue", "sum"),
                           loss=("credit_loss", "sum"),
                           funding=("funding_cost", "sum"),
                           contribution=("contribution", "sum"))
    g["gmv_share"]    = g["gmv"] / g["gmv"].sum()
    g["revenue_rate"] = g["revenue"] / g["gmv"]
    g["loss_rate"]    = g["loss"] / g["gmv"]
    g["funding_rate"] = g["funding"] / g["gmv"]
    g["margin"]       = g["contribution"] / g["gmv"]
    g["profit_share"] = g["contribution"] / g["contribution"].sum()
    cols = ["loans", "gmv", "gmv_share", "avg_fico", "bad_rate",
            "revenue_rate", "loss_rate", "funding_rate", "margin", "profit_share"]
    return g[cols].round(3)

**Reading the tables:** `bad_rate` = share of loans delinquent or charged off; `loss_rate`, `revenue_rate`,
`funding_rate`, `margin` = % of GMV; `profit_share` = share of total contribution dollars.

## 1. Portfolio health & trajectory
### 1.1 Headline numbers

In [ ]:
total = loans_clean[["loan_amount", "revenue", "credit_loss", "funding_cost", "contribution"]].sum()
print(f"Loans: {len(loans_clean):,}   Users: {loans_clean['user_id'].nunique():,}   GMV: ${total['loan_amount']:,.0f}")
print(f"Revenue:      {total['revenue'] / total['loan_amount']:.1%} of GMV")
print(f"Credit loss:  {total['credit_loss'] / total['loan_amount']:.1%} of GMV")
print(f"Funding cost: {total['funding_cost'] / total['loan_amount']:.1%} of GMV")
print(f"Contribution: {total['contribution'] / total['loan_amount']:.1%} of GMV  (${total['contribution']:,.0f})")

### 1.2 Trend by origination quarter

In [ ]:
by_quarter = summarize(loans_clean, "quarter")
by_quarter

In [ ]:
(by_quarter["gmv"] / 1000).plot(kind="bar", color="#2a78d6", title="GMV by origination quarter ($K)")
plt.show()

GMV grew every quarter, from ~$459K (Q3-2024) to ~$771K (Q4-2025). Average FICO is flat (~652), so growth hasn't
come from lowering credit quality. Loss and margin **fall** for recent quarters only because those loans are young
(Part 1, 4.1): they haven't defaulted *or* paid most of their interest yet. Use the older quarters to judge
steady-state economics: **~2–3% loss and ~7–8% margin.**

**Q4-2024 stands out:** 10% of its loans went bad, double the other quarters, and it happens in every vertical
(travel 35%, fashion 11%). Holiday-season loans performed worse — the Q4-2025 vintage is worth watching.

### 1.3 Where is the growth coming from? (2025-H2 vs 2024-H2)

In [ ]:
loans_clean["half"] = np.where(loans_clean["origination_date"] < "2025-01-01", "2024H2",
                np.where(loans_clean["origination_date"] < "2025-07-01", "2025H1", "2025H2"))

growth = loans_clean.pivot_table(index="vertical", columns="half", values="loan_amount", aggfunc="sum")
growth["growth"] = growth["2025H2"] / growth["2024H2"] - 1
growth.round(2).sort_values("growth", ascending=False)

In [ ]:
growth_p = loans_clean.pivot_table(index="product_type", columns="half", values="loan_amount", aggfunc="sum")
growth_p["growth"] = growth_p["2025H2"] / growth_p["2024H2"] - 1
growth_p.round(2)

- **Fashion is the fastest-growing vertical (+84%)**; electronics (+45%) and fitness (+54%) follow.
  Healthcare and auto parts are flat.
- **Interest-bearing installments drive the growth (+64%)**; Split Pay and 0% APR are flat.

## 2. Risk segmentation

### 2.1 By vertical

In [ ]:
by_vertical = summarize(loans_clean, "vertical").sort_values("margin")
by_vertical

In [ ]:
# Risk vs return: each dot is a vertical, dot size = GMV
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(by_vertical["loss_rate"] * 100, by_vertical["revenue_rate"] * 100,
           s=by_vertical["gmv"] / 1500, color="#2a78d6", alpha=0.7)
for name, row in by_vertical.iterrows():
    offset = (8, -14) if name == "fitness" else (8, 4)   # avoid overlapping home_furnishing
    ax.annotate(name, (row["loss_rate"] * 100, row["revenue_rate"] * 100),
                xytext=offset, textcoords="offset points")
ax.set_xlabel("Credit loss (% of GMV)")
ax.set_ylabel("Revenue (% of GMV)")
ax.set_title("Risk vs return by vertical (dot size = GMV)")
plt.show()

- **Travel** has by far the highest risk — 21% of loans are delinquent or charged off (others: 4–5%) and loss is
  3.5% of GMV — but also the highest revenue (13.4%). It still earns a healthy 8.7% margin and is the **largest
  source of profit (24%)**.
- **Fashion** is the weakest: lowest revenue (6.9%, small tickets), second-highest loss (2.5%), and the lowest margin
  (3.8%). It is 51% of all loans but only 10% of profit — and it's the fastest-growing vertical.
- **Healthcare** and **home furnishing** combine low loss with high revenue — best risk-adjusted segments.

### 2.2 Check: does the vertical ranking hold on older (seasoned) loans?

In [ ]:
seasoned = loans_clean[loans_clean["origination_date"] <= "2025-06-30"]
summarize(seasoned, "vertical").sort_values("margin")

On loans originated by Jun-2025 (≥ 6 months on book) the picture is the same: fashion lowest margin,
travel highest loss but still profitable, healthcare/home furnishing strongest. The ranking is not a maturity artifact.

### 2.3 By product

In [ ]:
summarize(loans_clean, "product_type")

- **Interest-bearing** loans are 58% of GMV but **82% of profit** (9.9% margin) — consumer interest pays for the risk.
- **Split Pay** (2.6%) and **0% APR** (3.4%) have thin margins: they earn only the merchant fee, and Split Pay's
  loss rate (2.2%) is as high as interest-bearing.

In [ ]:
# 0% APR is supposed to be paid for by a higher merchant fee. Is it? (same merchants only)
promo_merchants = loans_clean[loans_clean["has_promo_program"]]
promo_merchants.groupby("product_type")["mdr"].mean().round(4)

The brief says 0% APR is subsidized by a **higher** MDR — but at the same merchants, 0% loans carry a **lower** MDR
(5.0%) than interest-bearing loans (5.4%). 0% APR looks under-priced: Affirm gives up the interest and doesn't
get a higher merchant fee in return.

### 2.4 By FICO band

In [ ]:
summarize(loans_clean, "fico_band").loc[["<580", "580-639", "640-699", "700-759", "760+", "Unknown"]]

Loss rises steadily as FICO falls (0.7% at 760+ → 3.4% below 580). Revenue also rises (higher APRs), so margin is
fairly flat across bands — but it is lowest below 580.

### 2.5 New vs. repeat customers

In [ ]:
summarize(loans_clean, "customer")

Repeat customers lose **~2.5x less** than new customers (1.0% vs 2.6% of GMV) and earn a higher margin.
Getting a customer to a second loan is valuable.

### 2.6 Drill-down: what drives the weak segments?

In [ ]:
# Vertical x product
summarize(loans_clean, ["vertical", "product_type"])[["loans", "gmv_share", "bad_rate", "loss_rate", "margin", "profit_share"]]

In [ ]:
# Vertical x FICO (below vs above 640)
loans_clean["fico_640"] = np.where(loans_clean["fico_score"] < 640, "<640",
                    np.where(loans_clean["fico_score"] >= 640, "640+", "Unknown"))
summarize(loans_clean[loans_clean["fico_640"] != "Unknown"], ["vertical", "fico_640"])[["loans", "bad_rate", "loss_rate", "margin"]]

In [ ]:
# Travel: how sensitive is the margin to the loss assumption? (Part 1, 4.2)
# low  = charged-off loss only; base = as given; high = every delinquent loan charges off
travel = loans_clean[(loans_clean["vertical"] == "travel") & (loans_clean["fico_640"] != "Unknown")].copy()
delinquent = travel["loan_status"].isin(["delinquent_30", "delinquent_60"])
travel["unpaid_principal"] = travel["loan_amount"] - (travel["total_paid"] - travel["interest_collected"])
travel["loss_low"]  = np.where(delinquent, 0, travel["credit_loss"])
travel["loss_high"] = np.where(delinquent, travel["unpaid_principal"], travel["credit_loss"])

t = travel.groupby("fico_640")[["loan_amount", "revenue", "funding_cost", "loss_low", "credit_loss", "loss_high"]].sum()
for case in ["loss_low", "credit_loss", "loss_high"]:
    t["margin_" + case] = (t["revenue"] - t["funding_cost"] - t[case]) / t["loan_amount"]
t.filter(like="margin").round(3)

In [ ]:
# Vertical x new/repeat
summarize(loans_clean, ["vertical", "customer"])[["loans", "bad_rate", "loss_rate", "margin"]]

- **Travel's risk is concentrated in sub-640 FICO borrowers:** 40% of them are delinquent or charged off and loss is
  6.3% of GMV, vs 9% and 1.4% for 640+. Travel interest-bearing loans: 26% bad rate.
  Sub-640 travel still shows a 6.7% margin, but it is fragile: if the delinquent loans charge off, it drops to
  ~0%. Travel 640+ stays at ~9–11% in every case.
- **Fashion's weakness is Split Pay and new customers:** fashion Split Pay earns a ~1% margin on 10% of GMV;
  new fashion customers earn 1.7% margin (repeat: 5.9%).
- In most other verticals, sub-640 borrowers are still profitable — risk is priced by the higher APR.

Note: all 160 corrupted duplicate loans found in Part 1 were travel loans — the travel data feed may need a closer look.

## 3. Summary of findings
**Portfolio health**
- $4.0M GMV, 7,134 loans, growing every quarter; credit quality (FICO) stable.
- Steady-state economics: revenue ~10%, loss ~2%, funding ~1% → **~7% contribution margin**.
- Growth is led by interest-bearing loans (good — highest margin) and fashion (bad — lowest margin).

**Underperforming segments**
| Segment | What's wrong | Evidence |
|---|---|---|
| Travel, FICO < 640 | Very high default; margin wiped out if delinquents charge off | 40% bad rate, 6.3% loss; margin 6.7% → ~0% |
| Fashion Split Pay | Tiny tickets, MDR-only revenue | ~1% margin on 10% of GMV |
| New customers in fashion | High loss, low revenue | 3.6% loss, 1.7% margin |
| 0% APR | Merchant fee lower than interest-bearing | 3.4% margin; MDR 5.0% vs 5.4% |

**Seasonality:** the holiday vintage (Q4-2024) had double the bad rate of other quarters, in every vertical.

**Strong segments:** healthcare and home furnishing (low loss, high revenue), interest-bearing installments,
repeat customers.